# CODLING Training on Real Data (Google Colab)
===============================================

This notebook demonstrates training CODLING on Colab with T4 GPU using a real huggingface dataset (e.g. `the_pile` or `tiny_shakespeare`).

**Contents:**
- Clone repository & import requirements
- Load real data from HuggingFace Datasets
- Tokenize real text using GPT-2
- Configure and Instantiate Trainer
- Train & Save Checkpoints via Google Drive

---

## 1. Setup and Clone Repository
First, we mount google drive to comfortably store our model checkpoints, and we pull the latest `codling` code.

In [ ]:
# Optional: Mount Google Drive if you want persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Clone the codling repository
!git clone https://github.com/RaheesAhmed/codling.git
%cd codling

# Install dependencies (datasets, transformers, etc.)
!pip install -r requirements.txt -q
!pip install datasets transformers -q

import sys
sys.path.insert(0, '/content/codling')


## 2. Global Imports and Checkpoint Directory

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

from codling.codling.model import CodlingConfig, CodlingForCausalLM
from codling.codling.trainer import Trainer

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Persistent checkpoint directory on your Drive
CHECKPOINT_DIR = '/content/drive/MyDrive/codling_real_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## 3. Load HuggingFace Dataset & Tokenizer
Let's load `tiny_shakespeare` as a fast proof of concept for real-world text modeling. You can easily replace `"TinyShakespeare"` with `"the_pile"`.

In [ ]:
# We will use tiny_shakespeare for a quick text demo
dataset = load_dataset("Trelis/tiny-shakespeare")

train_data = dataset['train']
val_data = dataset['test']

print(f"Loaded {len(train_data)} training splits and {len(val_data)} validation splits.")

# Load GPT-2 Tokenizer padding config
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["Text"], truncation=True, max_length=512, padding="max_length")

print("Tokenizing dataset...")
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=["Text"])
tokenized_val = val_data.map(tokenize_function, batched=True, remove_columns=["Text"])

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")


## 4. Model Instantiation
We specify `CodlingConfig` and initialize our linear-time `CodlingForCausalLM`.

In [ ]:
# Model size configurations. Let's make an ~18M parameter model that trains incredibly fast.
config = CodlingConfig(
    vocab_size=len(tokenizer),
    d_model=256,
    n_layers=6,
    d_state=64,
    ssm_type='s4', # s4 or mamba
)

model = CodlingForCausalLM(config)
model.to(device)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Created Codling model with {params / 1e6:.2f} Million Parameters.")

## 5. Prepare the Dataloaders & Custom Trainer Loop
We implement the PyTorch training run and save.

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import torch.nn.functional as F
import gc

train_loader = DataLoader(tokenized_train, batch_size=8, shuffle=True)
val_loader = DataLoader(tokenized_val, batch_size=8)

optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
num_epochs = 3

print("Starting Training on Real Data...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        # The model causal LM wrapper accepts 'labels' directly
        labels = input_ids.clone()
        
        optimizer.zero_grad()
        outputs = model(input_ids, labels=labels)
        loss = outputs['loss']
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        progress.set_postfix({'loss': loss.item()})
        
    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Avg Train Loss: {avg_train_loss:.4f}")
    
    # Save Model Checkpoint to Google Drive
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"codling_model_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")
    gc.collect()
    torch.cuda.empty_cache()

## 6. Generation / Inference
Let's see what the model generates after training!

In [ ]:
model.eval()
prompt = "ROMEO:\nWhat a beautiful morning to"

input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

print("--- Input ---")
print(prompt)
print("\n--- Generated ---")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids, 
        max_new_tokens=50,
        temperature=0.8,
        top_k=40
    )
    
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))